In [ ]:
#| default_exp cfg_node

In [ ]:
#| hide
from nbdev.showdoc import *

# Parameters and Services

> Extend the camera publisher with runtime-configurable parameters and ROS2 services for capture and reset.

## Overview

The `CamPublisher` from notebook 02 sets parameters at launch time and never changes them. In real-world scenarios you want to tweak exposure, FPS, or resolution while the node is running — without restarting it.

This notebook adds:

- **Runtime parameter updates** — change exposure, FPS, resolution on the fly via `ros2 param set`
- **Capture service** — grab a single frame and save it to disk on demand
- **Reset service** — reinitialize the camera without restarting the node

## Imports

In [ ]:
#| export
#| hide
import sys
import os
import cv2
import rclpy
from rclpy.node import Node
from rclpy.executors import ExternalShutdownException
from rclpy.parameter import Parameter
from rclpy.callback_groups import ReentrantCallbackGroup
from cv_bridge import CvBridge
from sensor_msgs.msg import Image
from std_srvs.srv import Trigger

## Import Camera from 02

We reuse the `Camera` wrapper from notebook 02. The `CamPublisher` class is not imported directly — instead, we rebuild the node here with additional parameter callbacks and services.

In [ ]:
#| export
#| echo: false
from ros2_cam_nbdev.cam_pub import Camera

ImportError: cannot import name 'CamConfig' from 'ros2_cam_nbdev.cam_pub' (/home/ramonpr/Github/ros2_cam_nbdev_feature_cams/ros2_cam_nbdev/cam_pub.py)

## CfgNode

`CfgNode` extends the basic publisher pattern with:

- Parameter callbacks for live reconfiguration
- A `capture` service to save a frame on demand
- A `reset` service to reinitialize the camera

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `video_source` | string | `'0'` | Camera index or stream URL |
| `frame_rate` | float | `30.0` | Publishing frequency in FPS |
| `topic_name` | string | `'/camera/image_raw'` | Topic to publish images on |
| `video_fourcc` | string | `'MJPG'` | Compression format |
| `resolution_width` | int | `640` | Output frame width |
| `resolution_height` | int | `480` | Output frame height |
| `exposure` | int | `-1` | Camera exposure (-1 = auto) |
| `capture_path` | string | `'captures'` | Directory to save captured frames |

In [ ]:
#| export
class CfgNode(Node):
    """ROS2 camera publisher with runtime parameters and capture/reset services."""
    def __init__(self):
        super().__init__('cfg_node')
        self._cb_group = ReentrantCallbackGroup()

        # Declare parameters
        self.declare_parameter('video_source', '0')
        self.declare_parameter('frame_rate', 30.0)
        self.declare_parameter('topic_name', '/camera/image_raw')
        self.declare_parameter('video_fourcc', 'MJPG')
        self.declare_parameter('resolution_width', 640)
        self.declare_parameter('resolution_height', 480)
        self.declare_parameter('exposure', -1)
        self.declare_parameter('capture_path', 'captures')

        # Read parameters
        self.source = self.get_parameter('video_source').value
        self.frame_rate = self.get_parameter('frame_rate').value
        self.topic_name = self.get_parameter('topic_name').value
        fourcc = self.get_parameter('video_fourcc').value
        width = self.get_parameter('resolution_width').value
        height = self.get_parameter('resolution_height').value
        self.exposure = self.get_parameter('exposure').value
        self.capture_path = self.get_parameter('capture_path').value

        # Initialize camera
        self.camera = Camera(self.source, fourcc, width, height)
        if self.exposure >= 0:
            self.camera.cap.set(cv2.CAP_PROP_EXPOSURE, self.exposure)

        # Publisher
        self.publisher_ = self.create_publisher(Image, self.topic_name, 20)

        # Timer
        period = 1.0 / self.frame_rate
        self.timer = self.create_timer(period, self._publish_frame)

        # Parameter callback
        self.add_on_set_parameters_callback(self._on_params_changed)

        # Services
        self.srv_capture = self.create_service(
            Trigger, 'capture', self._handle_capture,
            callback_group=self._cb_group
        )
        self.srv_reset = self.create_service(
            Trigger, 'reset', self._handle_reset,
            callback_group=self._cb_group
        )

        # CvBridge
        self.bridge = CvBridge()
        self.frame_count = 0
        self._last_frame = None

        self.get_logger().info(
            f'CfgNode ready — source={self.source}, fps={self.frame_rate}, '
            f'topic="{self.topic_name}"'
        )

    def _publish_frame(self):
        """Capture a frame and publish it."""
        ret, frame = self.camera.read()
        if not ret:
            self.get_logger().warn('Failed to read frame')
            return

        self._last_frame = frame.copy()
        msg = self.bridge.cv2_to_imgmsg(frame, encoding='bgr8')
        self.publisher_.publish(msg)

        if self.frame_count % 10 == 0:
            self.get_logger().info(f'Published frame {self.frame_count}')
        self.frame_count += 1

    def _on_params_changed(self, params):
        """Handle runtime parameter changes."""
        from rcl_interfaces.msg import SetParametersResult
        for param in params:
            if param.name == 'exposure' and param.value >= 0:
                self.camera.cap.set(cv2.CAP_PROP_EXPOSURE, param.value)
                self.exposure = param.value
                self.get_logger().info(f'Exposure set to {param.value}')
            elif param.name == 'frame_rate' and param.value > 0:
                self.frame_rate = param.value
                self.timer.cancel()
                self.timer = self.create_timer(1.0 / self.frame_rate, self._publish_frame)
                self.get_logger().info(f'Frame rate set to {param.value} FPS')
            elif param.name in ('resolution_width', 'resolution_height'):
                w = self.get_parameter('resolution_width').value
                h = self.get_parameter('resolution_height').value
                self.camera.set_resolution(w, h)
                self.get_logger().info(f'Resolution set to {w}x{h}')
        return SetParametersResult(successful=True)

    def _handle_capture(self, request, response):
        """Capture a single frame and save it to disk."""
        if self._last_frame is None:
            response.success = False
            response.message = 'No frame available yet'
            return response

        os.makedirs(self.capture_path, exist_ok=True)
        filename = os.path.join(self.capture_path, f'capture_{self.frame_count}.png')
        cv2.imwrite(filename, self._last_frame)

        response.success = True
        response.message = f'Saved to {filename}'
        self.get_logger().info(response.message)
        return response

    def _handle_reset(self, request, response):
        """Reinitialize the camera."""
        self.camera.release()
        fourcc = self.get_parameter('video_fourcc').value
        width = self.get_parameter('resolution_width').value
        height = self.get_parameter('resolution_height').value
        self.camera = Camera(self.source, fourcc, width, height)

        response.success = True
        response.message = 'Camera reinitialized'
        self.get_logger().info(response.message)
        return response

    def destroy_node(self):
        """Release the camera before destroying the node."""
        self.camera.release()
        super().destroy_node()

### show_doc

In [ ]:
show_doc(CfgNode)

---

[source](https://github.com/Ramon-PR/ros2_cam_nbdev/blob/main/ros2_cam_nbdev/cfg_node.py#L26){target="_blank" style="float:right; font-size:smaller"}

### CfgNode

```python

def CfgNode(
    
):


```

*ROS2 camera publisher with runtime parameters and capture/reset services.*

## Entry Point

In [ ]:
#| export
def main(args=None):
    """Initialize ROS2, spin the CfgNode, and handle shutdown."""
    rclpy.init(args=args)
    node = CfgNode()

    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        node.get_logger().info('Node stopped by user or external shutdown')
    finally:
        node.destroy_node()
        rclpy.try_shutdown()

In [ ]:
#| exporti
if not hasattr(sys, 'ps1'):
    if __name__ == "__main__":
        main()

## Usage

First, export the notebook to generate the Python module:

``` bash
uv run nbdev-export
```

This creates `ros2_cam_nbdev/cfg_node.py`.

### Method 1 — Run directly with Python

The fastest way to test. No workspace or compilation needed.

**Terminal 1** — run the node:

``` bash
uv run python -m ros2_cam_nbdev.cfg_node
```

**Terminal 2** — check the topic is publishing:

``` bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

### Change parameters at runtime

``` bash
ros2 param set /cfg_node exposure 100
```

``` bash
ros2 param set /cfg_node frame_rate 15.0
```

``` bash
ros2 param set /cfg_node resolution_width 1280
ros2 param set /cfg_node resolution_height 720
```

### Call services

``` bash
ros2 service call /capture std_srvs/srv/Trigger
```

``` bash
ros2 service call /reset std_srvs/srv/Trigger
```

### Method 2 — Full ROS2 workflow

The standard ROS2 way: create a workspace, package, compile, and run.

**Step 1** — Source ROS2:

``` bash
source /opt/ros/$ROS_DISTRO/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 2** — Create the workspace:

``` bash
zsh scripts/generate_ros2_ws.zsh new_ros2_ws
```

**Step 3** — Create the package:

``` bash
zsh scripts/mk_ros2_pkg.sh new_ros2_ws my_camera_pkg camera_opencv.txt
```

> This creates `new_ros2_ws/src/my_camera_pkg/` with ROS2 dependencies
> read from `scripts/pkg_dependencies/camera_opencv.txt`.

**Step 4** — Copy the node into the package:

``` bash
cp ros2_cam_nbdev/cfg_node.py new_ros2_ws/src/my_camera_pkg/my_camera_pkg/
```

**Step 5** — Compile:

``` bash
bash scripts/compile_pkg.sh new_ros2_ws my_camera_pkg
```

> `compile_pkg.sh` generates `setup.py` automatically (entry points are
> discovered from any file with `def main(args=None):`) and runs `colcon build`.

**Step 6** — Source the workspace overlay:

``` bash
source new_ros2_ws/install/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 7** — Run the node:

``` bash
ros2 run my_camera_pkg cfg_node
```

**Terminal 2** — verify and visualize:

``` bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

## Visualize

With the node running, open a second terminal and visualize the images:

``` sh
ros2 run rqt_image_view rqt_image_view
```

Select `/camera/image_raw` from the dropdown.

For more visualization options (rviz2, Foxglove), see `08_visualization`.

## Next Steps

This node handles parameters and services, but what about hardware that needs careful init/cleanup? In `04_lifecycle_node` we explore managed lifecycle for such cases.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()